In [13]:
import transformers
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from peft import LoraConfig, get_peft_model, TaskType
from pyfaidx import Fasta
import pandas as pd
import torch



In [2]:
model_name = "zhihan1996/DNABERT-2-117M"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True
)

c:\Users\admin\anaconda3\envs\dnabert2_cftr\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Explicitly passing a `revision` is encouraged when loading a configuration with custom code to ensure no malicious code has been contributed in a newer revision.
Explicitly passing a `revision` is encouraged when loading a model with custom code to ensure no malicious code has been contributed in a newer revision.
C:\Users\admin/.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT-2-117M\7bce263b15377fc15361f52cfab88f8b586abda0\bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of the model checkpoint at zhihan1996/DNABERT-2-11

In [3]:
class DNABERTClassifier(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
        self.classifier = nn.Linear(768, 2)
    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        outputs = self.base(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs 
        )
        cls = outputs[0][:, 0, :]
        return self.classifier(cls)

In [65]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

PeftModelForFeatureExtraction(
  (base_model): LoraModel(
    (model): DNABERTClassifier(
      (base): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(4096, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertUnpadAttention(
                (self): BertUnpadSelfAttention(
                  (dropout): Dropout(p=0.0, inplace=False)
                  (Wqkv): Linear(
                    in_features=768, out_features=2304, bias=True
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=8, bias=

In [4]:
model = DNABERTClassifier(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["Wqkv"],  
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)

model = get_peft_model(model, lora_config)

In [5]:
genome = Fasta("D:\CFTR\Homo_sapiens_CFTR_sequence.fa")

In [6]:
df = pd.read_csv(r"D:\CFTR\final_cftr_dataset.csv")
df.head()

,Variant cDNA name,variant determination,hgvs_genomic_grch38,chr,pos,ref,alt,cause,Name,Canonical SPDI,dbSNP ID,Variant type,Germline classification
0,c.1521_1523del,CF-causing,NC_000007.14:g.117559592_117559594del,chr7,117559590,ATCT,A,1,NaN,NaN,NaN,NaN,NaN
1,c.1624G>T,CF-causing,NC_000007.14:g.117587778G>T,chr7,117587778,G,T,1,NaN,NaN,NaN,NaN,NaN
2,c.1652G>A,CF-causing,NC_000007.14:g.117587806G>A,chr7,117587806,G,A,1,NaN,NaN,NaN,NaN,NaN
3,c.3909C>G,CF-causing,NC_000007.14:g.117652877C>G,chr7,117652877,C,G,1,NaN,NaN,NaN,NaN,NaN
4,c.3718-2477C>T,CF-causing,NC_000007.14:g.117639961C>T,chr7,117639961,C,T,1,NaN,NaN,NaN,NaN,NaN


In [29]:
df = df[~df["Variant type"].isin(["Haplotype"])]

In [8]:
START = 117287120


In [9]:
def extract_window(genome, pos, START, window=200):
    local_pos = pos - START
    
    start = local_pos - window
    end = local_pos + window
    
    seq = genome["7"][start:end].seq.upper()
    return seq

In [57]:
def apply_mutation(seq, ref, alt, window=200):
    center = window

    # ensure strings
    ref = str(ref)
    alt = str(alt)

    # CASE 1: insertion (ref empty)
    if ref == "":
        mutated = seq[:center] + alt + seq[center:]

    # CASE 2: substitution or deletion
    else:
        seq_ref = seq[center:center+len(ref)]
        
        if seq_ref != ref:
            # optional debug
            # print("REF mismatch:", seq_ref, ref)
            return None
        
        mutated = seq[:center] + alt + seq[center+len(ref):]

    return mutated

In [58]:
def create_input(ref_seq, mut_seq):
    return ref_seq + "[SEP]" + mut_seq

In [59]:
def tokenize_input(tokenizer, input_seq):
    tokens = tokenizer(
        input_seq,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    
    return {
        "input_ids": tokens["input_ids"].squeeze(),
        "attention_mask": tokens["attention_mask"].squeeze()
    }

In [60]:
import torch

def process_row(row, tokenizer, genome, START, window=200):
    pos = row["pos"].astype(float).astype(int)
    ref = row["ref"]
    alt = row["alt"]
    label = int(row["cause"])
    #print(ref, alt)

    # Step 1: extract
    ref_seq = extract_window(genome, pos, START, window)
    if ref_seq is None or len(ref_seq) == 0:
        return None

    # Step 2: mutate
    mut_seq = apply_mutation(ref_seq, ref, alt, window)
    if mut_seq is None:
        return None

    # Step 3: combine
    input_seq = create_input(ref_seq, mut_seq)

    # Step 4: tokenize
    tokens = tokenize_input(tokenizer, input_seq)

    return {
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"],
        "labels": torch.tensor(label)
    }

In [61]:
def build_dataset(df, tokenizer, genome, START):
    data = []
    
    for i in range(len(df)):
        row = df.iloc[i]
        item = process_row(row, tokenizer, genome, START)
        
        if item is not None:
            data.append(item)
    
    return data

In [62]:
from torch.utils.data import DataLoader

dataset = build_dataset(df, tokenizer, genome, START)

loader = DataLoader(dataset, batch_size=4, shuffle=True)

In [63]:
row = df.iloc[0]

item = process_row(row, tokenizer, genome, START)

print(item["input_ids"].shape)
print(item["labels"])

torch.Size([512])
tensor(1)


In [68]:

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()


In [ ]:
for epoch in range(3):
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 0.06686223298311234
Epoch: 1 Loss: 1.4590797424316406
Epoch: 2 Loss: 0.05789671838283539


In [70]:
for epoch in range(7):
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 0.0638323426246643
Epoch: 1 Loss: 0.04094588756561279
Epoch: 2 Loss: 0.04551565647125244
Epoch: 3 Loss: 0.042460791766643524
Epoch: 4 Loss: 0.03529631346464157
Epoch: 5 Loss: 0.032941192388534546
Epoch: 6 Loss: 0.028048651292920113
